In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, when, expr, round
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType

# Find the exact PySpark version on your computer
spark_version = pyspark.__version__

# Start the Spark session
spark = SparkSession.builder \
    .appName("CentralKitchenPipeline") \
    .config("spark.jars.packages", f"org.apache.spark:spark-sql-kafka-0-10_2.13:{spark_version}") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

26/03/20 14:13:39 WARN Utils: Your hostname, o7 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/20 14:13:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/hduser/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/student/.ivy2/cache
The jars for the packages stored in: /home/student/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4321a813-47be-4988-ae01-3981d4d906ed;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;3.5.5 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;3.5.5 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.0.4 in central
	found org.apache.commons#commons-pool2

In [2]:
import json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType
from pyspark.sql.functions import col, from_json, when, expr, round

# Open the config file and load the settings
with open('config.json', 'r') as file:
    config = json.load(file)

# Save the settings into variables
kafka_servers = config.get("kafka_bootstrap_servers", "localhost:9092")
kitchen_topic = config.get("kitchen_station_events", "kitchen_station_events")
dispatch_topic = config.get("dispatch_events", "dispatch_events")
kitchen_path = config.get("kitchen_data_path", "./output/kitchen_data")
dispatch_path = config.get("dispatch_data_path", "./output/dispatch_data")
kitchen_checkpoint_path = config.get("kitchen_checkpoint_path", "./output/checkpoints/kitchen")
dispatch_checkpoint_path = config.get("dispatch_checkpoint_path", "./output/checkpoints/dispatch")
max_truck_temp = config.get("max_truck_temp", 5.0)
min_cook_temp = config.get("min_cook_temp", 75.0)
min_weight_drop = config.get("min_weight_drop", 2.0)

# Build the data schemas
ingredient_schema = StructType([
    StructField("item", StringType(), True),
    StructField("amount_kg", DoubleType(), True)
])

kitchen_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("station_id", StringType(), True),
    StructField("recipe_id", StringType(), True),
    StructField("action", StringType(), True),
    StructField("weight_kg", DoubleType(), True),
    StructField("ingredients", ArrayType(ingredient_schema), True),
    StructField("temperature_celsius", DoubleType(), True),
    StructField("event_timestamp", StringType(), True)
])

dispatch_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("canteen_id", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("action", StringType(), True),
    StructField("truck_temp_celsius", DoubleType(), True),
    StructField("event_timestamp", StringType(), True)
])

# Connect to the local Kafka server
kitchen_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_servers) \
    .option("subscribe", kitchen_topic) \
    .load()

dispatch_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_servers) \
    .option("subscribe", dispatch_topic) \
    .load()

# Parse the text data
kitchen_parsed = kitchen_stream.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), kitchen_schema).alias("data")) \
    .select("data.*")

# Add up the ingredients to find the starting weight
kitchen_with_start_weight = kitchen_parsed.withColumn(
    "expected_start_weight",
    round(expr("aggregate(ingredients, 0D, (acc, item) -> acc + item.amount_kg)"), 2)
)

# Add the status flags using the new config variables
kitchen_checked = kitchen_with_start_weight.withColumn(
    "sensor_status",
    when(col("weight_kg").isNull() | col("temperature_celsius").isNull(), "SENSOR_FAILURE")
    .otherwise("OK")
).withColumn(
    "temperature_status",
    when((col("action") == "COOKING") & (col("temperature_celsius") < min_cook_temp), "LOW_HEAT_WARNING")
    .otherwise("OK")
).withColumn(
    "weight_status",
    when(col("weight_kg").isNull(), "UNKNOWN")
    .when((col("action").isin("PREPARING", "COOKING", "PACKING")) & ((col("expected_start_weight") - col("weight_kg")) >= min_weight_drop), "SUSPICIOUS_DROP")
    .otherwise("OK")
)

# Parse the text and find errors for the dispatch data
dispatch_parsed = dispatch_stream.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), dispatch_schema).alias("data")) \
    .select("data.*")

dispatch_checked = dispatch_parsed.withColumn(
    "sensor_status",
    when(col("truck_temp_celsius").isNull(), "SENSOR_FAILURE")
    .otherwise("OK")
).withColumn(
    "temperature_status",
    when(col("truck_temp_celsius") > max_truck_temp, "WARM_TRUCK_WARNING")
    .otherwise("OK")
)

# Save the organized data into storage
kitchen_query = kitchen_checked.writeStream \
    .format("parquet") \
    .option("path", kitchen_path) \
    .option("checkpointLocation", kitchen_checkpoint_path) \
    .start()

dispatch_query = dispatch_checked.writeStream \
    .format("parquet") \
    .option("path", dispatch_path) \
    .option("checkpointLocation", dispatch_checkpoint_path) \
    .start()

print("Spark Streaming is running. Press Ctrl+C to stop.")
spark.streams.awaitAnyTermination()

26/03/20 14:13:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/03/20 14:13:45 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Spark Streaming is running. Press Ctrl+C to stop.


26/03/20 14:13:45 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/03/20 14:13:45 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/home/student/de-venv/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/student/de-venv/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [3]:
import json
from pyspark.sql import SparkSession

with open('config.json', 'r') as file:
    config = json.load(file)
    
# Start a basic Spark session
spark = SparkSession.builder \
    .appName("ReadSavedData") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

kitchen_path = config.get("kitchen_data_path", "./output/kitchen_data")
dispatch_path = config.get("dispatch_data_path", "./output/dispatch_data")

print("Reading Kitchen Data")
try:
    # Open the files and show the data
    kitchen_data = spark.read.parquet(kitchen_path)
    # Sort by time so you can see the batch progress
    kitchen_data.orderBy("event_timestamp").show(200, truncate=False)
except Exception as e:
    print("No kitchen data found.")

print("\nReading Dispatch Data...")
try:
    dispatch_data = spark.read.parquet(dispatch_path)
    dispatch_data.orderBy("event_timestamp").show(200, truncate=False)
except Exception as e:
    print("No dispatch data found.")

26/03/20 14:14:06 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Reading Kitchen Data
+------------+----------+----------+------------+---------+---------+-------------------------------------------------------------------+-------------------+----------------+---------------------+--------------+------------------+---------------+
|event_id    |batch_id  |station_id|recipe_id   |action   |weight_kg|ingredients                                                        |temperature_celsius|event_timestamp |expected_start_weight|sensor_status |temperature_status|weight_status  |
+------------+----------+----------+------------+---------+---------+-------------------------------------------------------------------+-------------------+----------------+---------------------+--------------+------------------+---------------+
|KIT-EVT-1001|BATCH-1001|prep_01   |fish_curry  |PREPARING|13.52    |[{white_fish, 6.76}, {curry_paste, 4.06}, {coconut_milk, 2.7}]     |21.5               |2026-03-15T06:01|13.52                |OK            |OK                |OK      